# PINN Dynamics — Research Notebook

**Physics-Informed Neural Networks for Dynamical System Identification**

This notebook is the reproducible research layer of the project. It benchmarks forward prediction, inverse parameter recovery, robustness to noise, and high-frequency training.

## 1. Research question

Can a PINN recover hidden physical parameters from sparse noisy observations while respecting the governing differential equation? We use the damped harmonic oscillator as a controlled testbed.

In [ ]:
import sys, os
sys.path.append('..')
import torch, numpy as np, matplotlib.pyplot as plt
from src.pinn import *
torch.manual_seed(7)
np.random.seed(7)

## 2. Physical model

We solve

$$m u'' + \mu u' + ku = 0,$$

with $u(0)=1$ and $u'(0)=0$. The analytical solution is used **only for evaluation**.

In [ ]:
cfg = OscillatorConfig(m=1.0, mu=4.0, k=400.0, t_end=1.0)
print(cfg)
print('damping rate:', cfg.delta)
print('natural frequency:', cfg.omega0)
print('damped frequency:', cfg.omega_d)

## 3. Forward PINN

The network learns $u_\theta(t)$. Automatic differentiation supplies $u'$ and $u''$, and the physics residual is

$$r(t)=mu''+\mu u'+ku.$$

In [ ]:
model = MLP(hidden=64, layers=4)
t_phys = torch.linspace(0, 1, 600).view(-1,1).requires_grad_(True)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
history=[]
for step in range(4000):
    opt.zero_grad()
    loss, parts = losses_forward(model, t_phys, cfg)
    loss.backward(); opt.step()
    if step % 100 == 0: history.append(float(loss))
print('final loss:', history[-1])

In [ ]:
t = torch.linspace(0,1,1000).view(-1,1)
pred=model(t).detach(); truth=exact_solution(t,cfg)
err=relative_l2(pred,truth).item()
plt.figure(figsize=(8,4)); plt.plot(t,truth,label='Analytical'); plt.plot(t,pred,label='PINN',ls='--'); plt.xlabel('t'); plt.ylabel('u(t)'); plt.title(f'Forward solution | relative L2 = {err:.2e}'); plt.legend(); plt.show()

## 4. Inverse PINN: learn damping

Now the damping coefficient is unknown. We constrain it to remain positive using a softplus parameterization and train on sparse noisy measurements plus the governing equation.

In [ ]:
t_obs, u_obs = make_observations(cfg, n=30, noise_std=0.05, seed=7)
model_inv = MLP(hidden=64, layers=4)
mu_param = PositiveParameter(1.0)
t_phys = torch.linspace(0,1,500).view(-1,1).requires_grad_(True)
opt = torch.optim.Adam(list(model_inv.parameters()) + list(mu_param.parameters()), lr=1e-3)
mu_hist=[]
for step in range(5000):
    opt.zero_grad()
    loss, parts = losses_inverse(model_inv,t_phys,t_obs,u_obs,cfg.m,cfg.k,mu_param)
    loss.backward(); opt.step()
    mu_hist.append(mu_param().item())
print(f'True mu = {cfg.mu:.4f}')
print(f'Estimated mu = {mu_param().item():.4f}')
print(f'Relative parameter error = {abs(mu_param().item()-cfg.mu)/cfg.mu:.2%}')

In [ ]:
plt.figure(figsize=(8,4)); plt.plot(mu_hist); plt.axhline(cfg.mu,ls='--',label='true mu'); plt.xlabel('training step'); plt.ylabel('estimated mu'); plt.title('Inverse PINN parameter convergence'); plt.legend(); plt.show()

## 5. Noise robustness

Repeat the inverse problem across increasing observation noise. The main metric is relative damping-parameter error.

In [ ]:
noise_results=[]
for noise in [0.0,0.01,0.05,0.1,0.2]:
    torch.manual_seed(7)
    m=MLP(hidden=64,layers=4); p=PositiveParameter(1.0)
    to,uo=make_observations(cfg,30,noise,seed=7)
    opt=torch.optim.Adam(list(m.parameters())+list(p.parameters()),lr=1e-3)
    tp=torch.linspace(0,1,500).view(-1,1).requires_grad_(True)
    for step in range(2500):
        opt.zero_grad(); loss,_=losses_inverse(m,tp,to,uo,cfg.m,cfg.k,p); loss.backward(); opt.step()
    est=p().item(); rel=abs(est-cfg.mu)/cfg.mu
    noise_results.append((noise,est,rel))
noise_results

## 6. High-frequency regime

Standard MLPs often have difficulty representing rapidly oscillating functions. We compare the vanilla network against Fourier-feature inputs as the natural frequency increases.

In [ ]:
freq_results=[]
for w0 in [20,40,80,120]:
    c=OscillatorConfig(mu=4,k=w0*w0)
    for name, Model in [('Vanilla',MLP),('Fourier',FourierPINN)]:
        torch.manual_seed(7)
        m=Model(hidden=64,layers=4)
        p=PositiveParameter(1.0)
        to,uo=make_observations(c,30,0.05,seed=7)
        tp=torch.linspace(0,1,700).view(-1,1).requires_grad_(True)
        opt=torch.optim.Adam(list(m.parameters())+list(p.parameters()),lr=1e-3)
        for step in range(3000):
            opt.zero_grad(); loss,_=losses_inverse(m,tp,to,uo,c.m,c.k,p); loss.backward(); opt.step()
        est=p().item(); freq_results.append((w0,name,est,abs(est-c.mu)/c.mu))
for row in freq_results: print(row)

## 7. Interpretation

The project should report three separate conclusions rather than one headline accuracy:

1. **Forward accuracy:** does the PINN reproduce the physical trajectory?
2. **Identifiability:** can the hidden damping coefficient be recovered from sparse observations?
3. **Optimization/generalization limits:** how do noise and frequency affect the inverse problem, and does Fourier encoding help?

The analytical solution is never required by the PINN itself; it is an evaluation oracle for this controlled benchmark.

## 8. What to report in the final README

Record the actual numbers generated by the experiments rather than hard-coding expected results. Include: forward relative L2 error, inverse damping estimate, parameter error at each noise level, and vanilla-vs-Fourier comparison at each frequency.